# day-12-model-bakeoff — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [10]:
# ---- Solution 4 ----
lenient = collections.defaultdict(list)
for r in rows:
    lenient[r["model"]].append(int(r["pred"] == r["gold"]))
print(f"{'model':14s} {'strict acc':>11} {'lenient acc':>12}")
for d in summary:
    strict = d["accuracy"]
    len_acc = np.mean(lenient[d["model"]])
    print(f"{d['model']:14s} {strict:>11.2f} {len_acc:>12.2f}   (+{len_acc-strict:.2f})")
print("\nS4: a model whose lenient >> strict was getting the answer RIGHT but the FORMAT wrong")
print("-- a cheap output parser recovers it. In this run the gap is ~0: Qwen's format is")
print("already clean, and the SmolLM2 outputs are wrong on content too, not just format.")
print("Rule stands: format failures are often fixable downstream; wrong labels are not.")

model           strict acc  lenient acc
Qwen2.5-0.5B          0.83         0.83   (+0.00)
SmolLM2-360M          0.17         0.17   (+0.00)
SmolLM2-135M          0.00         0.00   (+0.00)

S4: a model whose lenient >> strict was getting the answer RIGHT but the FORMAT wrong
-- a cheap output parser recovers it. In this run the gap is ~0: Qwen's format is
already clean, and the SmolLM2 outputs are wrong on content too, not just format.
Rule stands: format failures are often fixable downstream; wrong labels are not.


In [11]:
# ---- Solution 6 ----
p135 = [r["pred"] for r in rows if r["model"] == "SmolLM2-135M"]
p360 = [r["pred"] for r in rows if r["model"] == "SmolLM2-360M"]
disagree = [(GOLD[i][0], GOLD[i][1], p135[i], p360[i])
            for i in range(len(GOLD)) if p135[i] != p360[i]]
print(f"{len(disagree)} tickets where the two SmolLM2 models disagree:")
for t, g, a, b in disagree:
    print(f"  gold={g:8s} 135M={str(a):8s} 360M={str(b):8s}  {t[:55]}")

12 tickets where the two SmolLM2 models disagree:
  gold=billing  135M=None     360M=other     I was double charged for my subscription and want a ref
  gold=bug      135M=None     360M=other     The export button does nothing when I click it.
  gold=account  135M=None     360M=other     I can't log in, my account says it's locked.
  gold=feature  135M=None     360M=other     Please add a dark mode to the dashboard.
  gold=other    135M=None     360M=other     What are your business hours?
  gold=billing  135M=None     360M=other     My invoice has the wrong VAT number.
  gold=bug      135M=None     360M=other     The app crashes on startup after the last update.
  gold=account  135M=None     360M=other     How do I change my email address?
  gold=feature  135M=None     360M=other     It would be great to have a Slack integration.
  gold=other    135M=None     360M=other     Your pricing page has a typo.
  gold=billing  135M=None     360M=other     I was billed after I cancelled last m

### Solutions 1, 2, 3, 5 (sketch)

- **S1:** put `Ticket: "..." -> billing` style examples in the system string for a `chat=False`
  `LocalHF("distilgpt2-fewshot", "distilgpt2", chat=False)`. Non-instruct models often *can*
  pattern-complete a label from 3 examples even though they ignore an instruction — that's the
  Day 05 few-shot lesson showing up in a model comparison.
- **S2:** add e.g. `"haiku": dict(pin=0.80, pout=4.00)` and a `ClaudeAPI` adapter. A cheap API
  model typically lands up-and-right of the local models: higher accuracy, higher (but still
  low) cost, and much lower latency than CPU inference.
- **S3:** `with ThreadPoolExecutor(max_workers=3) as ex: list(ex.map(...))` around `complete`;
  throughput = `len(GOLD) / wall_time`. On CPU the models contend for cores so throughput
  barely rises; on a GPU or an API it scales.
- **S5:** from a typical run only `SmolLM2-360M` clears 0.85 accuracy, and its CPU p95 is > 1.5s
  — so *nothing* qualifies as-is. Next moves: better prompt (Day 05/06), few-shot, a small
  fine-tune (Week 3), or accept an API model. This is exactly how a bake-off tells you to
  escalate.

### Answer key
1. So the bake-off code is model-agnostic: you can add or swap local and API models without
   touching the scoring, aggregation, or plotting code.
2. They have different fixes — format failures are often recoverable with a downstream parser
   or constrained decoding; wrong answers need a better prompt, more data, or a better model.
   A blended score hides which one is hurting.
3. Report both rankings, test more prompt variants and more items, and don't pick a model
   until the ranking is stable. A ranking that flips on wording isn't measuring capability.
4. Batch/stream requests; cache the system prompt; quantize the model; move off CPU; raise
   `max_new_tokens` ceiling down; or shorten the prompt.
5. The model usually knows the right answer but won't emit the required format. Not a problem
   if you control the downstream parser (recover it cheaply); a problem if the raw string is
   the product or a strict schema is required — then fine-tune / constrain decoding.
6. It concentrates the genuinely ambiguous or hard cases; those are the highest-value items to
   label carefully and add to the eval set (and a place where model choice actually matters).
7. 12 items gives a very wide confidence interval on accuracy (±~15 points); the winner is
   suggestive, not established. Scale the gold set to 50–200 before committing.